# Phase 1: Production RAG
This notebook sets up the Vector Database and tests the retrieval pipeline.

In [2]:
!pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers pypdf huggingface_hub langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 k

In [3]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Libraries imported successfully!")

/tmp/ipykernel_773/3363890380.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Libraries imported successfully!


In [6]:
# CELL 3: Load and Chunk PDFs

# We simulate the folder here for Colab
os.makedirs('data/raw_pdfs', exist_ok=True)

loader = PyPDFDirectoryLoader("data/raw_pdfs")
documents = loader.load()
print(f"Loaded {len(documents)} pages from PDFs.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks.")

Loaded 300 pages from PDFs.
Created 2041 chunks.


In [7]:
# CELL 4: Embeddings and Vector DB Setup
# Using BAAI/bge-large-en-v1.5 which is very fast and fits in Colab RAM
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

# Create ChromaDB Vector Store
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("Vector database created and saved locally!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Vector database created and saved locally!


In [8]:
# CELL 5: Test Retrieval
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

query = "What is the main topic of the documents?"
results = retriever.invoke(query)

for idx, res in enumerate(results):
    print(f"--- Document {idx+1} ---")
    print(res.page_content)
    print("-" * 30)

--- Document 1 ---
scheduled meetings ofthe Federal Open Market Committee•Appendix C contains information on the Federal Reserve’s audited financial statements as wellas reviews conducted by the Office of Inspector General and the Government Account-ability Office•Appendix D presents information on the budgets for the Board and Reserve Banks and oncurrency-related costs•Appendix E summarizes policy actions of the Board of Governors•Appendix F lists litigation, both pending and resolved, that the Board of Governors was aparty
------------------------------
--- Document 2 ---
annual report. A broader set of eco-nomic and financial developments are discussed in section 2, “Monetary Policy and EconomicDevelopments,” with the discussion that follows concerning surveillance of economic and financialdevelopments focused on financial stability. The full range of activities associated with supervision
------------------------------
--- Document 3 ---
Report on Banking Applications Activity, whi